# 01 — Data Collection & Cleaning

Loads the customer dataset, inspects its structure, checks missing values, removes missing records, and creates the transformed customer-level features used throughout the project.

## Project workflow

1. Problem understanding
2. Data collection
3. Data cleaning
4. Feature transformation
5. Exploratory analysis
6. Predictive modelling
7. Customer segmentation

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")


## Load dataset

In [ ]:
# Update this path if your CSV is stored elsewhere.
DATA_PATH = "../data/customer.csv"
df = pd.read_csv(DATA_PATH)
df.head()


## Dataset information

In [ ]:
df.info()

## Missing-value check

In [ ]:
df.isnull().sum()

## Remove missing records

In [ ]:
df = df.dropna()
df.shape

df.isnull().sum()

## Feature transformation

The original notebook derives age, age group, first visit recency and an `OfferAccepted` indicator.

In [ ]:
def isLeap(year):
  Leap = None
  if year % 400==0:
    Leap = True
  elif year % 100 == 0:
    Leap = False
  elif year % 4 == 0:
    Leap = True
  else:
    Leap = False
  return Leap


def get_LastVisit(date):
  Months = [31,28,31,30,31,30,31,31,30,31,30,31]
  today = [1,1,2023]
  d,m,y = list(map(int,date.split('-')))
  Months[1] = 29 if isLeap(y) else 28
  d1 = Months[m-1]-d + (today[0]-1)
  d2 = sum(Months[m:])+sum(Months[:today[0]-1])
  d3 = sum([366 if isLeap(y) else 365 for y in range(y+1,today[2])])
  diff = d1+d2+d3
  return diff

def getAgeGroup(age):
  ageGroup = ''
  if age <= 18:
    ageGroup = 'Teenager'
  elif age >18 and age<= 30:
    ageGroup = 'Bachelors'
  elif age >30 and age<=50:
    ageGroup  = 'Adults'
  elif age >50 and age<=70:
    ageGroup = 'Seniors'
  elif age>=70 and age<100:
    ageGroup = 'Retired'
  else:
    ageGroup = 'Expired'
  return ageGroup

def AcceptOffer(responses):
  Accepted = 0
  if 1 in responses:
    Accepted = 1

  return Accepted

def get_LastVisit(date):
    if isinstance(date, str):
        Months = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
        today = [1, 1, 2023]
        d, m, y = list(map(int, date.split('-')))
        Months[1] = 29 if isLeap(y) else 28
        d1 = Months[m - 1] - d + (today[0] - 1)
        d2 = sum(Months[m:]) + sum(Months[:today[0] - 1])
        d3 = sum([366 if isLeap(y) else 365 for y in range(y + 1, today[2])])
        diff = d1 + d2 + d3
        return diff
    else:
        return 0  # Return 0 or handle the case when the date is not a string

# Rest of the code...
df['Age'] = df.apply(lambda row: 2023- int(row.Year_Birth), axis=1)
df['FirstVisit'] = df.apply(lambda row : get_LastVisit(row.Dt_Customer),axis=1)
df['AgeGroup'] = df.apply(lambda row: getAgeGroup(row.Age), axis = 1)
df.rename(columns = {'Response': 'AcceptedCmpN'}, inplace = True)
df["OfferAccepted"] = df.apply(lambda row : AcceptOffer([row.AcceptedCmp1,row.AcceptedCmp2,row.AcceptedCmp3,row.AcceptedCmp4,row.AcceptedCmp5,row.AcceptedCmpN]), axis = 1)

df['Age'] = df.apply(lambda row: 2023 - int(row.Year_Birth), axis=1)
df['FirstVisit'] = df.apply(lambda row: get_LastVisit(row.Dt_Customer), axis=1)
df['AgeGroup'] = df.apply(lambda row: getAgeGroup(row.Age), axis=1)
df.rename(columns={'Response': 'AcceptedCmpN'}, inplace=True)
df["OfferAccepted"] = df.apply(
    lambda row: AcceptOffer([
        row.AcceptedCmp1, row.AcceptedCmp2, row.AcceptedCmp3,
        row.AcceptedCmp4, row.AcceptedCmp5, row.AcceptedCmpN
    ]), axis=1
)


## Select useful business features

In [ ]:
df2 = df[['Age', 'AgeGroup','Education', 'Marital_Status', 'Income', 'Kidhome', 'Teenhome', 'FirstVisit', 'Recency', 'MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth', "OfferAccepted", 'Complain']]
df2

## Save cleaned dataset for later notebooks

In [ ]:
df2.to_csv("../data/cleaned_customer.csv", index=False)
df2.head()